# BRFSS 2024 Cardiac Dataset Preparation

This notebook keeps **19 predictors and 1 binary target**. It decodes BRFSS-specific response values, gives the columns readable names, checks the result, and saves one clean CSV for later modeling. The source is the [CDC 2024 BRFSS public dataset](https://www.cdc.gov/brfss/annual_data/annual_2024.html).

**Workflow**

`Raw XPT file` → `select 20 variables` → `clean survey codes` → `validate` → `save processed CSV`

> Missing predictor values are intentionally left as `NaN`. Imputation, scaling, encoding, and class-balancing belong inside the later training pipeline so information from the test set cannot leak into training.

---

## 1. Setup

Only three libraries are needed here: **NumPy** for missing values, **pandas** for table operations, and **Path** for clear, portable file locations.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

### File locations

The raw survey and processed output are kept separate. The source XPT file is only read; every transformation is written to a new CSV in the `processed` folder. This makes the workflow reproducible and protects the original data.

In [2]:

RAW_PATH = Path("../../data/LLCP2024/raw/LLCP2024.XPT")

PROCESSED_DIR = Path("../../data/LLCP2024/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "brfss_2024_cardiac_clean.csv"

---

## 2. Load and inspect the raw survey

BRFSS is distributed as a SAS XPORT (`.XPT`) file. The completed run below loaded **457,670 survey responses and 301 columns**, so the raw table is far wider than the small feature set needed for this project.

In [3]:
df = pd.read_sas(
    RAW_PATH,
    format="xport"
)

print("Dataset shape:", df.shape)

Dataset shape: (457670, 301)


### A quick structural check

The first five rows confirm that the file opened correctly. `df.info()` then reveals the data types and memory footprint: most survey fields are numeric, a few are stored as objects, and the full table occupies a little over **1 GB** in memory.

In [4]:
display(df.head())
df.info()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000001',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0
1,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000002',2.024000e+09,1.0,...,4.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
2,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000003',2.024000e+09,1.0,...,4.0,2.0,1.0,1.000000e+02,2.0,1.400000e+03,1.0,NaN,NaN,2.0
3,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000004',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
4,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000005',2.024000e+09,1.0,...,3.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,2.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Columns: 301 entries, _STATE to _AIDTST4
dtypes: float64(296), object(5)
memory usage: 1.0+ GB


---

## 3. Select the variables that answer the project question

The selected fields cover the main areas that may relate to cardiac health:

| Area | Variables |
|---|---|
| Demographics | age and sex |
| Health status | BMI, general health, and poor physical/mental health days |
| Lifestyle and care | checkups, activity, smoking, and alcohol use |
| Medical history | diabetes, stroke, and kidney disease |
| Social and access factors | education, income, employment, walking difficulty, personal doctor, and cost barriers |
| Target | `_MICHD`, a CDC-derived indicator of reported coronary heart disease or heart attack |


> **Leakage guard:** `CVDINFR4` and `CVDCRHD4` are deliberately excluded because CDC uses them to construct `_MICHD`. Including them as predictors would give the model part of the answer.

In [45]:
selected_columns = [
    "_AGE80",
    "_SEX",
    "_BMI5",
    "GENHLTH",
    "PHYSHLTH",
    "MENTHLTH",
    "CHECKUP1",
    "_TOTINDA",
    "DIABETE4",
    "CVDSTRK3",
    "CHCKDNY2",
    "_SMOKER3",
    "ALCDAY4",
    "_EDUCAG",
    "INCOME3",
    "EMPLOY1",
    "DIFFWALK",
    "PERSDOC3",
    "MEDCOST1",
    "_MICHD",
]

### Create a focused working copy

Keeping only the required columns reduces the table to **457,670 rows × 20 columns**. Calling `.copy()` makes `cardiac` an independent working dataset, so cleaning it does not accidentally modify the full raw DataFrame.

In [6]:
cardiac = df[selected_columns].copy()

In [7]:
print(cardiac.shape)

(457670, 20)


---

## 4. Audit the values before changing them

BRFSS codes must be interpreted **one variable at a time**. For example, `88` means “none” for health-day questions, while other fields use different values for “don't know” or “refused.” The frequency tables expose these codes before any replacement is made.

The first missing-value scan measures only values already stored as `NaN`. At this stage, BMI has about **9.40%** missing and alcohol frequency about **8.57%**; these percentages rise later when special non-answers are correctly converted to `NaN`.

In [8]:
for column in cardiac.columns:
    print("\n", "=" * 60)
    print(column)
    print("=" * 60)

    print(
        cardiac[column]
        .value_counts(dropna=False)
        .sort_index()
        .head(50)
    )


_AGE80
_AGE80
18.0     3773
19.0     4009
20.0     4159
21.0     4290
22.0     4197
23.0     4502
24.0     4764
25.0     4926
26.0     4629
27.0     4695
28.0     4774
29.0     4685
30.0     5635
31.0     4597
32.0     5287
33.0     5420
34.0     5346
35.0     5924
36.0     5512
37.0     5654
38.0     6001
39.0     5933
40.0     6891
41.0     5462
42.0     6651
43.0     6294
44.0     6096
45.0     6496
46.0     5774
47.0     6021
48.0     5694
49.0     5764
50.0     6887
51.0     5467
52.0     6910
53.0     7336
54.0     8109
55.0     7870
56.0     7057
57.0     7028
58.0     7256
59.0     7466
60.0     9066
61.0     7534
62.0     9177
63.0     8786
64.0     9040
65.0    10386
66.0     9124
67.0     9810
Name: count, dtype: int64

_SEX
_SEX
1.0    217507
2.0    240163
Name: count, dtype: int64

_BMI5
_BMI5
1200.0    1
1205.0    2
1208.0    1
1211.0    5
1213.0    1
1216.0    1
1217.0    1
1219.0    1
1220.0    2
1221.0    3
1222.0    1
1226.0    1
1227.0    2
1230.0    1
1231.0    1
1

In [9]:
missing_summary = (
    cardiac
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
)

display(missing_summary.to_frame())

,missing_percent
_BMI5,9.403500
ALCDAY4,8.569712
DIFFWALK,4.079795
INCOME3,2.025258
_MICHD,1.137501
EMPLOY1,0.721699
MEDCOST1,0.001311
CHCKDNY2,0.001311
GENHLTH,0.001092
PHYSHLTH,0.001092


---

## 5. Build a reliable target

`_MICHD` represents whether a respondent reported coronary heart disease or a heart attack. Its valid codes are mapped into a model-friendly target:

| Original code | Meaning | New value |
|---:|---|---:|
| `1` | Reported cardiac disease | `1` |
| `2` | Did not report cardiac disease | `0` |
| Missing/other | Target is unknown | Row removed |

The raw count shows **5,206 unknown targets**. Removing them leaves **452,464 respondents** with an unambiguous outcome, and the target is renamed to `cardiac_disease`.

In [10]:
cardiac["_MICHD"].value_counts(dropna=False)

_MICHD
2.0    410126
1.0     42338
NaN      5206
Name: count, dtype: int64

In [11]:
cardiac = cardiac[
    cardiac["_MICHD"].isin([1, 2])
].copy()

In [12]:
cardiac["_MICHD"] = cardiac["_MICHD"].map({
    1: 1,
    2: 0,
})

In [13]:
cardiac = cardiac.rename(
    columns={"_MICHD": "cardiac_disease"}
)

In [14]:
cardiac["cardiac_disease"].value_counts()

cardiac_disease
0    410126
1     42338
Name: count, dtype: int64

---

## 6. Clean the numerical features

The audit already shows age values from **18 to 80**, so age can stay as recorded; 80 is the top-coded value for respondents aged 80 or above. Sex contains only codes `1` and `2`, so it also needs no missing-code cleanup and will later be treated as categorical—not as a quantity.

### BMI: restore the decimal point

`_BMI5` stores BMI with two implied decimal places. Dividing by 100 changes values such as `2855` into `28.55`. The before-and-after summaries confirm a final observed range of **12.00 to 99.84**.

In [15]:
cardiac["_BMI5"].describe()

count    410281.000000
mean       2855.161969
std         657.777866
min        1200.000000
25%        2414.000000
50%        2744.000000
75%        3175.000000
max        9984.000000
Name: _BMI5, dtype: float64

In [16]:
cardiac["_BMI5"] = cardiac["_BMI5"] / 100

In [17]:
cardiac["_BMI5"].describe()

count    410281.000000
mean         28.551620
std           6.577779
min          12.000000
25%          24.140000
50%          27.440000
75%          31.750000
max          99.840000
Name: _BMI5, dtype: float64

---

## 7. Decode categorical survey responses

### General health

Valid responses run from `1 = Excellent` to `5 = Poor`. Codes `7` (“don't know”) and `9` (“refused”) are not health levels, so they become `NaN`. The cleaned output contains **1,201 missing responses** and no remaining 7s or 9s.

In [18]:
cardiac["GENHLTH"].value_counts()


GENHLTH
3.0    154392
2.0    144984
4.0     66479
1.0     63938
5.0     21470
7.0       860
9.0       338
Name: count, dtype: int64

In [19]:
cardiac["GENHLTH"] = cardiac["GENHLTH"].replace({
    7: np.nan,
    9: np.nan,
})

In [20]:
cardiac["GENHLTH"].value_counts(
    dropna=False
).sort_index()

GENHLTH
1.0     63938
2.0    144984
3.0    154392
4.0     66479
5.0     21470
NaN      1201
Name: count, dtype: int64

### Poor physical and mental health days

These questions count unhealthy days during the last 30 days. Their unusual code `88` means **zero days**, not missing; `77` and `99` mean unknown or refused and therefore become `NaN`. The resulting values stay within the meaningful **0–30 day** range.

In [21]:
cardiac["PHYSHLTH"] = cardiac["PHYSHLTH"].replace({
    88: 0,
    77: np.nan,
    99: np.nan,
})

In [22]:
cardiac["PHYSHLTH"].describe()

count    441918.000000
mean          4.565465
std           8.886671
min           0.000000
25%           0.000000
50%           0.000000
75%           4.000000
max          30.000000
Name: PHYSHLTH, dtype: float64

In [23]:
cardiac["PHYSHLTH"] = cardiac["PHYSHLTH"].where(
    cardiac["PHYSHLTH"].between(0, 30)
)

In [24]:
cardiac["MENTHLTH"] = cardiac["MENTHLTH"].replace({
    88: 0,
    77: np.nan,
    99: np.nan,
})

### Time since the last routine checkup

Only `7` (unknown) and `9` (refused) are removed. Code `8` means the respondent has **never** had a routine checkup, which is meaningful information and is kept as its own category.

In [25]:
cardiac["CHECKUP1"] = cardiac["CHECKUP1"].replace({
    7: np.nan,
    9: np.nan,
})

### Physical activity

`_TOTINDA` is a CDC-calculated leisure-time activity indicator. Codes `1` and `2` are valid answers; code `9` is not, so the **1,194** occurrences shown in the audit are converted to `NaN`.

In [26]:
cardiac["_TOTINDA"].value_counts(dropna=False)

_TOTINDA
1.0    346912
2.0    104358
9.0      1194
Name: count, dtype: int64

In [27]:
cardiac["_TOTINDA"] = cardiac["_TOTINDA"].where(
    cardiac["_TOTINDA"].isin([1, 2])
)

### Diabetes, stroke, and kidney disease

Unknown and refused responses (`7` and `9`) become missing. Diabetes keeps all four informative states—including pregnancy-only diabetes and prediabetes—while stroke and kidney disease retain their yes/no categories. This preserves detail instead of forcing every condition into an oversimplified binary value.

In [28]:
cardiac["DIABETE4"] = cardiac["DIABETE4"].replace({
    7: np.nan,
    9: np.nan,
})

In [29]:
cardiac["CVDSTRK3"] = cardiac["CVDSTRK3"].replace({
    7: np.nan,
    9: np.nan,
})

In [30]:
cardiac["CHCKDNY2"] = cardiac["CHCKDNY2"].replace({
    7: np.nan,
    9: np.nan,
})

### Smoking status

`_SMOKER3` groups respondents as current daily, current occasional, former, or never smokers (`1–4`). Code `9` is invalid for analysis and becomes missing. The audit finds **31,382** such responses. These groups remain categorical because their code numbers are labels, not measured amounts.

In [31]:
cardiac["_SMOKER3"].value_counts(dropna=False)

_SMOKER3
4.0    256497
3.0    118181
1.0     32628
9.0     31382
2.0     13776
Name: count, dtype: int64

In [32]:
cardiac["_SMOKER3"] = cardiac["_SMOKER3"].where(
    cardiac["_SMOKER3"].isin([1, 2, 3, 4])
)

### Alcohol frequency: convert mixed codes into one measure

`ALCDAY4` mixes weekly and monthly answers in the same column, so its raw numbers cannot be used directly:

- `101–199`: days per week, converted to an approximate 30-day frequency
- `201–299`: days during the past 30 days
- `888`: no alcohol, converted to `0`
- `777` or `999`: unknown/refused, converted to `NaN`

After conversion, the feature has an intuitive **0–30 days per month** range; the observed median is **1 day**.

In [33]:
def convert_alcohol_days_per_month(value):
    if pd.isna(value):
        return np.nan

    value = int(value)

    if value == 888:
        return 0.0

    if value in (777, 999):
        return np.nan

    if 101 <= value <= 199:
        days_per_week = value - 100

        return days_per_week * 30 / 7

    if 201 <= value <= 299:
        return float(value - 200)

    return np.nan

In [34]:
cardiac["ALCDAY4"] = cardiac["ALCDAY4"].apply(
    convert_alcohol_days_per_month
)

In [35]:
cardiac["ALCDAY4"].describe()

count    409468.000000
mean          4.652083
std           7.936238
min           0.000000
25%           0.000000
50%           1.000000
75%           5.000000
max          30.000000
Name: ALCDAY4, dtype: float64

In [36]:
cardiac["ALCDAY4"] = cardiac["ALCDAY4"].where(
    cardiac["ALCDAY4"].between(0, 30)
)

### Education, income, employment, function, and access to care

This final cleaning block applies each field's own valid-code rules:

| Feature | Values retained | Values changed to `NaN` |
|---|---|---|
| Education | grouped levels `1–4` | everything else |
| Income | ordered groups `1–11` | `77`, `99`, and out-of-range values |
| Employment | statuses `1–8` | `9` and out-of-range values |
| Walking difficulty | yes/no | `7`, `9` |
| Personal doctor | one, multiple, or none | `7`, `9` |
| Medical cost barrier | yes/no | `7`, `9` |

The important idea is consistent: genuine categories are preserved, while non-answers become missing values.

In [37]:
# Education
cardiac["_EDUCAG"] = cardiac["_EDUCAG"].where(
    cardiac["_EDUCAG"].isin([1, 2, 3, 4])
)


# Income
cardiac["INCOME3"] = cardiac["INCOME3"].replace({
    77: np.nan,
    99: np.nan,
})

cardiac["INCOME3"] = cardiac["INCOME3"].where(
    cardiac["INCOME3"].between(1, 11)
)


# Employment
cardiac["EMPLOY1"] = cardiac["EMPLOY1"].replace({
    9: np.nan,
})

cardiac["EMPLOY1"] = cardiac["EMPLOY1"].where(
    cardiac["EMPLOY1"].between(1, 8)
)


# Difficulty walking
cardiac["DIFFWALK"] = cardiac["DIFFWALK"].replace({
    7: np.nan,
    9: np.nan,
})


# Personal doctor
cardiac["PERSDOC3"] = cardiac["PERSDOC3"].replace({
    7: np.nan,
    9: np.nan,
})


# Medical cost barrier
cardiac["MEDCOST1"] = cardiac["MEDCOST1"].replace({
    7: np.nan,
    9: np.nan,
})

---

## 8. Replace codebook names with readable names

CDC names are useful when checking the official codebook, but names such as `poor_physical_health_days` and `medical_cost_barrier` are much easier to understand in charts, models, and reports. Only the labels change here—the data values stay exactly the same.

In [38]:
rename_map = {
    "_AGE80": "age",
    "_SEX": "sex",
    "_BMI5": "bmi",
    "GENHLTH": "general_health",
    "PHYSHLTH": "poor_physical_health_days",
    "MENTHLTH": "poor_mental_health_days",
    "CHECKUP1": "last_checkup",
    "_TOTINDA": "physical_activity",
    "DIABETE4": "diabetes_status",
    "CVDSTRK3": "stroke_history",
    "CHCKDNY2": "kidney_disease",
    "_SMOKER3": "smoking_status",
    "ALCDAY4": "alcohol_days_per_month",
    "_EDUCAG": "education_level",
    "INCOME3": "income_level",
    "EMPLOY1": "employment_status",
    "DIFFWALK": "difficulty_walking",
    "PERSDOC3": "personal_doctor",
    "MEDCOST1": "medical_cost_barrier",
    "_MICHD": "cardiac_disease",
}

cardiac = cardiac.rename(columns=rename_map)

---

## 9. Validate the cleaned dataset

### Exact duplicate patterns

The check finds **1,063 identical rows**, but they are not automatically deleted. With only 20 survey fields and no respondent ID in this subset, two different people can legitimately give the same answers. Removing them would risk discarding real observations.

In [39]:
duplicate_count = cardiac.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 1063


### Missing values after decoding

This second audit is the meaningful one: it includes both original missing values and survey non-answers converted during cleaning. Income has the most missing data (**18.91%**), followed by alcohol frequency (**9.50%**) and BMI (**9.32%**). The target has no missing values.

> These predictor gaps are retained. A later model pipeline can learn imputation values from the training data only.

In [40]:
missing_report = pd.DataFrame({
    "missing_count": cardiac.isna().sum(),
    "missing_percent": (
        cardiac.isna().mean() * 100
    ),
})

missing_report = missing_report.sort_values(
    "missing_percent",
    ascending=False,
)

display(missing_report.round(2))

,missing_count,missing_percent
income_level,85558,18.91
alcohol_days_per_month,42996,9.50
bmi,42183,9.32
smoking_status,31382,6.94
difficulty_walking,19969,4.41
poor_physical_health_days,10546,2.33
employment_status,8143,1.80
poor_mental_health_days,7757,1.71
last_checkup,5094,1.13
personal_doctor,4408,0.97


### Target balance

Reported cardiac disease is the minority outcome: **42,338 respondents (9.36%)** are positive, compared with **410,126 (90.64%)** negative. This imbalance should influence model choice and evaluation metrics, but it should not be artificially changed in the saved dataset.

In [41]:
class_counts = cardiac[
    "cardiac_disease"
].value_counts()

class_percent = (
    cardiac["cardiac_disease"]
    .value_counts(normalize=True)
    .mul(100)
)

print(class_counts)
print()
print(class_percent.round(2))

cardiac_disease
0    410126
1     42338
Name: count, dtype: int64

cardiac_disease
0    90.64
1     9.36
Name: proportion, dtype: float64


### Numerical range check

The summary confirms the intended units and boundaries: age is **18–80**, both unhealthy-day measures are **0–30**, alcohol frequency is **0–30 days per month**, and BMI is now expressed with its decimal point. This is a fast way to catch missed survey codes or conversion errors.

In [42]:
numeric_check = cardiac[
    [
        "age",
        "bmi",
        "poor_physical_health_days",
        "poor_mental_health_days",
        "alcohol_days_per_month",
    ]
].describe()

display(numeric_check)

,age,bmi,poor_physical_health_days,poor_mental_health_days,alcohol_days_per_month
count,452464.000000,410281.000000,441918.000000,444707.000000,409468.000000
mean,55.015104,28.551620,4.565465,4.396906,4.652083
std,18.120707,6.577779,8.886671,8.338029,7.936238
min,18.000000,12.000000,0.000000,0.000000,0.000000
25%,40.000000,24.140000,0.000000,0.000000,0.000000
50%,58.000000,27.440000,0.000000,0.000000,1.000000
75%,70.000000,31.750000,4.000000,5.000000,5.000000
max,80.000000,99.840000,30.000000,30.000000,30.000000


### Categorical value check

The final frequency tables verify that each categorical feature contains only its intended codes plus `NaN`. This catches subtle mistakes that a numerical summary cannot—for example, an unhandled refusal code appearing as a real category.

In [43]:
categorical_columns = [
    "sex",
    "general_health",
    "last_checkup",
    "physical_activity",
    "diabetes_status",
    "stroke_history",
    "kidney_disease",
    "smoking_status",
    "education_level",
    "income_level",
    "employment_status",
    "difficulty_walking",
    "personal_doctor",
    "medical_cost_barrier",
]
for column in categorical_columns:
    print("\n", column)

    print(
        cardiac[column]
        .value_counts(dropna=False)
        .sort_index()
    )


 sex
sex
1.0    214790
2.0    237674
Name: count, dtype: int64

 general_health
general_health
1.0     63938
2.0    144984
3.0    154392
4.0     66479
5.0     21470
NaN      1201
Name: count, dtype: int64

 last_checkup
last_checkup
1.0    365772
2.0     39328
3.0     21293
4.0     18248
8.0      2729
NaN      5094
Name: count, dtype: int64

 physical_activity
physical_activity
1.0    346912
2.0    104358
NaN      1194
Name: count, dtype: int64

 diabetes_status
diabetes_status
1.0     64586
2.0      3365
3.0    372683
4.0     11059
NaN       771
Name: count, dtype: int64

 stroke_history
stroke_history
1.0     20102
2.0    431367
NaN       995
Name: count, dtype: int64

 kidney_disease
kidney_disease
1.0     23236
2.0    427574
NaN      1654
Name: count, dtype: int64

 smoking_status
smoking_status
1.0     32628
2.0     13776
3.0    118181
4.0    256497
NaN     31382
Name: count, dtype: int64

 education_level
education_level
1.0     26417
2.0    114133
3.0    119463
4.0    190252
Na

---

## 10. Save the processed dataset

The cleaned table is saved without a pandas index, producing **452,464 rows × 20 columns** at `data/LLCP2024/processed/brfss_2024_cardiac_clean.csv`. This file becomes the single input for later EDA and modeling notebooks.

In [44]:
cardiac.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Final shape: {cardiac.shape}")

Saved to: ..\..\data\LLCP2024\processed\brfss_2024_cardiac_clean.csv
Final shape: (452464, 20)


---

## Preparation summary

The final file contains **19 predictors and one complete binary target**. Survey non-answers have been converted to `NaN`, BMI and alcohol frequency now use understandable units, and all columns have readable names.

**What happens next—inside the modeling workflow:**

1. Split the data into training and test sets using target stratification.
2. Learn imputation, scaling, and one-hot encoding from the training data only.
3. Handle class imbalance within training or cross-validation if needed.
4. Evaluate the chosen model once on the untouched test set.

> **Final takeaway:** this CSV is clean and auditable, while learned preprocessing is intentionally postponed to prevent data leakage.